In [ ]:
import os
import json
import ssl

import numpy as np
import pandas as pd

import cv2

import tensorflow as tf

from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.layers import Dense,GRU,Dropout,GlobalAveragePooling2D,Input
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.metrics import AUC

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import recall_score,f1_score,confusion_matrix,roc_auc_score,balanced_accuracy_score

ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
IMG_SIZE, NUM_FEATURES, SEQ_LEN = 224, 2048, 20
VID_DIR = "../dataset/train_sample_videos"
YUNET_PATH = "../model/face_detection_yunet.onnx"

In [3]:
np.random.seed(42)
tf.random.set_seed(42) 

## Preprocessing (face-focused)

In [4]:
# Build the InceptionV3 feature extractor (ImageNet weights)
def build_feature_extractor():
    base_model = InceptionV3(
        weights="imagenet",
        include_top=False,
        pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    outputs = base_model(preprocess_input(inputs))
    feature_extractor = Model(inputs, outputs)
    return feature_extractor

In [5]:
# Crop the center square (fallback when no face is found)
def crop_center_square(frame):
    height, width = frame.shape[:2]
    square_size = min(height, width)
    start_x = (width - square_size) // 2
    start_y = (height - square_size) // 2
    
    cropped_frame = frame[
    start_y:start_y + square_size,
    start_x:start_x + square_size
    ]

    return cropped_frame

In [6]:
# Detect and crop the largest face
def crop_face(frame, detector, margin=0.30):
    
    height, width = frame.shape[:2]
    detector.setInputSize((width, height))
    _, faces = detector.detect(frame)

    # Use center crop if no face is found
    if faces is None or len(faces) == 0:
        face_crop = crop_center_square(frame)

    else:

        # Find largest face
        largest_face = faces[0]

        for face in faces:

            current_area = face[2] * face[3]
            largest_area = largest_face[2] * largest_face[3]

            if current_area > largest_area:
                largest_face = face

        x, y, face_width, face_height = largest_face[:4]

        center_x = x + face_width / 2
        center_y = y + face_height / 2

        crop_size = max(face_width, face_height)
        crop_size = crop_size * (1 + margin)

        x_start = int(max(0, center_x - crop_size / 2))
        y_start = int(max(0, center_y - crop_size / 2))

        x_end = int(min(width, center_x + crop_size / 2))
        y_end = int(min(height, center_y + crop_size / 2))

        face_crop = frame[
            y_start:y_end,
            x_start:x_end
        ]

        # Use center crop if face crop fails
        if face_crop.size == 0:
            face_crop = crop_center_square(frame)

    # Resize image
    face_crop = cv2.resize(
        face_crop,
        (IMG_SIZE, IMG_SIZE)
    )

    # Convert BGR to RGB
    face_crop = cv2.cvtColor(
        face_crop,
        cv2.COLOR_BGR2RGB
    )

    return face_crop

In [7]:
# Extract features for a list of videos -> (features, masks)
def extract_face_frames(video_path, detector):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )
    if total_frames > 0:
        frame_step = max(1, total_frames // SEQ_LEN)
    else:
        frame_step = 1
    frames = []
    frame_index = 0
    while len(frames) < SEQ_LEN:
        success, frame = cap.read()
        if not success:
            break

        # Select evenly spaced frames
        if frame_index % frame_step == 0:

            face_frame = crop_face(frame, detector)
            frames.append(face_frame)

        frame_index += 1

    cap.release()

    return np.array(frames)

In [8]:
# Extract features and masks for a list of videos
def extract_features(video_list, feature_extractor):
    face_detector = cv2.FaceDetectorYN_create(YUNET_PATH, "", (320, 320), score_threshold=0.6)

    number_of_videos = len(video_list)
    features = np.zeros((number_of_videos, SEQ_LEN, NUM_FEATURES), dtype="float32")
    masks = np.zeros((number_of_videos, SEQ_LEN), dtype="bool")

    for index in range(number_of_videos):
        video_path = os.path.join(VID_DIR, video_list[index])
        frames = extract_face_frames(video_path, face_detector)
        number_of_frames = min(SEQ_LEN, len(frames))

        # Skip videos that could not be read (a few files are missing on disk)
        if number_of_frames > 0:
            frame_features = feature_extractor.predict(frames[:number_of_frames], verbose=0)
            features[index, :number_of_frames] = frame_features
            masks[index, :number_of_frames] = True

    return features, masks

## Load data, split, and extract features
Labels: FAKE=1, REAL=0. Stratified 75/25 split.

In [9]:
# Load dataset metadata
with open(os.path.join(VID_DIR, "metadata.json")) as file:
    metadata = json.load(file)

In [10]:
video_ids = []
labels = []

# Keep only videos that actually exist on disk (a few metadata entries are missing)
available_videos = set(os.listdir(VID_DIR))

# Store video names and labels
for video_name, info in metadata.items():
    if video_name.endswith(".mp4") and video_name in available_videos:
        video_ids.append(video_name)
        if info["label"] == "FAKE":
            labels.append(1)
        else:
            labels.append(0)

labels = np.array(labels)

print(
    "Videos:", len(video_ids),
    "| FAKE:", np.sum(labels),
    "| REAL:", np.sum(labels == 0)
)

Videos: 394 | FAKE: 320 | REAL: 74


In [11]:
train_videos, test_videos = train_test_split(video_ids,test_size=0.25,random_state=42,stratify=labels)

In [12]:
# Build feature extractor
feature_extractor = build_feature_extractor()

In [13]:
features, masks = extract_features(video_ids, feature_extractor)

KeyboardInterrupt: 

## Prepare features for each video

In [14]:
# Map each video name to its row in the features array
video_to_index = {}
for index in range(len(video_ids)):
    video_to_index[video_ids[index]] = index

# Get the features, masks, and labels for a list of videos
def get_data_for_videos(video_list):
    selected_features = []
    selected_masks = []
    selected_labels = []
    for video_name in video_list:
        position = video_to_index[video_name]
        selected_features.append(features[position])
        selected_masks.append(masks[position])
        selected_labels.append(labels[position])
    return np.array(selected_features), np.array(selected_masks), np.array(selected_labels)

In [ ]:
# Get the labels of the training videos (needed for a balanced split)
train_labels = []
for video_name in train_videos:
    train_labels.append(labels[video_to_index[video_name]])

In [ ]:
# Split the training videos into training and validation videos
train_videos, validation_videos = train_test_split(
    train_videos,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

In [ ]:
print(f"Train videos      : {len(train_videos)}")
print(f"Validation videos : {len(validation_videos)}")
print(f"Test videos       : {len(test_videos)}")

Train videos      : 236
Validation videos : 59
Test videos       : 99


## Build the train, validation, and test sets

In [16]:
# Build the input arrays for each split
X_train, M_train, y_train = get_data_for_videos(train_videos)
X_validation, M_validation, y_validation = get_data_for_videos(validation_videos)
X_test, M_test, y_test = get_data_for_videos(test_videos)

print("Train      :", X_train.shape, "| FAKE:", int(np.sum(y_train)), "REAL:", int(np.sum(y_train == 0)))
print("Validation :", X_validation.shape, "| FAKE:", int(np.sum(y_validation)), "REAL:", int(np.sum(y_validation == 0)))
print("Test       :", X_test.shape, "| FAKE:", int(np.sum(y_test)), "REAL:", int(np.sum(y_test == 0)))

Train      : (236, 20, 2048) | FAKE: 192 REAL: 44
Validation : (59, 20, 2048) | FAKE: 48 REAL: 11
Test       : (99, 20, 2048) | FAKE: 80 REAL: 19


## Class weights

In [17]:
# Calculate class weights to balance the FAKE / REAL difference
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train
)
class_weights = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weights)

Class weights: {0: 2.6818181818181817, 1: 0.6145833333333334}


## Build the GRU model
The model takes two inputs: the frame features and a mask that says which frames are real.

In [ ]:
# Reset the seed so the model trains the same way each run
tf.keras.utils.set_random_seed(42)

# Build the GRU model (two inputs: features + mask)
feature_input = Input(shape=(SEQ_LEN, NUM_FEATURES))
mask_input = Input(shape=(SEQ_LEN,), dtype="bool")

x = GRU(64, return_sequences=True)(feature_input, mask=mask_input)
x = GRU(32)(x)
x = Dropout(0.4)(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=[feature_input, mask_input], outputs=output)



model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 20, 2048)]           0         []                            
                                                                                                  
 input_4 (InputLayer)        [(None, 20)]                 0         []                            
                                                                                                  
 gru (GRU)                   (None, 20, 64)               405888    ['input_3[0][0]',             
                                                                     'input_4[0][0]']             
                                                                                                  
 gru_1 (GRU)                 (None, 32)                   9408      ['gru[0][0]']           

In [ ]:
model.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=[AUC(name="auc")]
)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=15,
    restore_best_weights=True
)

In [ ]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=6,
    min_lr=1e-5
)

In [ ]:
# Train the model
history = model.fit(
    [X_train, M_train],
    y_train,
    validation_data=([X_validation, M_validation], y_validation),
    epochs=120,
    batch_size=16,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr],
    verbose=2
)

print("Trained for", len(history.history["loss"]), "epochs")
print("Best validation AUC:", round(max(history.history["val_auc"]), 3))

Epoch 1/120
15/15 - 3s - loss: 0.8150 - auc: 0.5065 - val_loss: 0.6445 - val_auc: 0.4602 - lr: 0.0010 - 3s/epoch - 206ms/step
Epoch 2/120
15/15 - 0s - loss: 0.7085 - auc: 0.5595 - val_loss: 0.7100 - val_auc: 0.7140 - lr: 0.0010 - 219ms/epoch - 15ms/step
Epoch 3/120
15/15 - 0s - loss: 0.6965 - auc: 0.5579 - val_loss: 0.6584 - val_auc: 0.7538 - lr: 0.0010 - 218ms/epoch - 15ms/step
Epoch 4/120
15/15 - 0s - loss: 0.7281 - auc: 0.4730 - val_loss: 0.6864 - val_auc: 0.7367 - lr: 0.0010 - 219ms/epoch - 15ms/step
Epoch 5/120
15/15 - 0s - loss: 0.6912 - auc: 0.5500 - val_loss: 0.6982 - val_auc: 0.8116 - lr: 0.0010 - 220ms/epoch - 15ms/step
Epoch 6/120
15/15 - 0s - loss: 0.6919 - auc: 0.5614 - val_loss: 0.6896 - val_auc: 0.7983 - lr: 0.0010 - 220ms/epoch - 15ms/step
Epoch 7/120
15/15 - 0s - loss: 0.6951 - auc: 0.5373 - val_loss: 0.6918 - val_auc: 0.8513 - lr: 0.0010 - 227ms/epoch - 15ms/step
Epoch 8/120
15/15 - 0s - loss: 0.6661 - auc: 0.6178 - val_loss: 0.6801 - val_auc: 0.8428 - lr: 0.0010 - 23

## Evaluate the model

In [20]:
# Predict on the test set (the model outputs the probability of FAKE)
predicted_probabilities = model.predict([X_test, M_test], verbose=0).ravel()

# Turn probabilities into labels (0.5 or higher means FAKE)
predicted_labels = (predicted_probabilities >= 0.5).astype(int)

# Calculate the metrics
roc_auc = roc_auc_score(y_test, predicted_probabilities)
recall = recall_score(y_test, predicted_labels)
f1 = f1_score(y_test, predicted_labels)
balanced_accuracy = balanced_accuracy_score(y_test, predicted_labels)

print("ROC-AUC           :", round(roc_auc, 3))
print("Recall            :", round(recall, 3))
print("F1 Score          :", round(f1, 3))
print("Balanced Accuracy :", round(balanced_accuracy, 3))

# Show the confusion matrix
print()
print("Confusion Matrix (rows = true REAL/FAKE, columns = predicted REAL/FAKE):")
print(confusion_matrix(y_test, predicted_labels))

ROC-AUC           : 0.847
Recall            : 0.95
F1 Score          : 0.921
Balanced Accuracy : 0.738

Confusion Matrix (rows = true REAL/FAKE, columns = predicted REAL/FAKE):
[[10  9]
 [ 4 76]]


In [21]:
# Save the trained model so app.py can use it
model.save("../model/deepfake_video_model_v2.h5")
print("Model saved to ../model/deepfake_video_model_v2.h5")

Model saved to ../model/deepfake_video_model_v2.h5


## Results

| Metric | Score |
|--------|-------|
| ROC-AUC | ~0.85 |
| Recall (FAKE) | ~0.95 |
| F1 score (FAKE) | ~0.92 |

**Why face cropping works:** deepfake artifacts live on the face. Using full frames, the
InceptionV3 features barely separated real from fake (AUC ~0.59). Cropping to the face
raised separability to AUC ~0.85, which is the main reason this model works.

The dataset has few REAL videos (74), so the model can sometimes flag a real video as fake.
More REAL training data is the best way to improve this further.